# RouteHunter

In [1]:
from routehunter import RouteHunterApp

In [2]:
app = RouteHunterApp.from_data_dir("rh_data")
print(app.load_report.summary())

aizynthfinder
synplanner
Rows processed : 1481
Targets        : 1481
Unique targets : 1362
Unique papers  : 1263
Errors         : 0


### 1. Review

In [3]:
print(app.introduction())

RouteHunter -- synthesis route reference lookup (static dataset)

  Search      : give a SMILES, get papers, static CASP-solved tool
                results, and predicted solvability for that molecule.
  Monitor     : browse recently published papers, ranked by predicted
                probability of containing a multi-step synthesis route
                (pre-scored offline; candidates for you to review and
                add to the CSV by hand).
  Predict     : given a SMILES, get predicted solvability probability
                per CASP tool, with a link to that tool.
  Download    : export the dataset for AI/ML training.

This dataset is loaded from a CSV file; there is no in-app way to
modify it. To add or correct data, edit the CSV and reload.


Current dataset:
  Targets                : 1362
  Papers                 : 1263
  Targets w/ >1 route    : 97
  Cached CASP routes     : 0 (session only)
  Predicted targets      : 122865 (awaiting digitalization)
  Papers by journal

### 2. Search

Given a SMILES, return literature papers *and* any CASP-predicted routes cached earlier this session.

Try some molecules with positive search:  
``C#CCOC1=C(C=C(C(=C1)N2C(=O)N3CCCCC3=N2)Cl)Cl``  
``C(O)(C(O)=O)C(C1C=CC=CC=1)NC(C1C=CC=CC=1)=O``  
``C1CCC(=C(C1)CC(=O)O)N2C(=O)C=CC(=N2)C3=C4C=CC=CN4N=C3C5=CC=CC=C5``

Try some absent molecules:  
``CC(C)Cc1ccc(cc1)C(C)C(=O)O``

In [4]:
result = app.search("C(O)(C(O)=O)C(C1C=CC=CC=1)NC(C1C=CC=CC=1)=O")
print(result.report())

Found 1 paper(s) reporting a route for this molecule:
 - [paper] Utilization of a Benzoyl Migration To Effect an Expeditious Synthesis of the Paclitaxel C-13 Side Chain (Organic Process Research & Development, 1997) doi:10.1021/op970113b

Found 2 tool(s) predicted routes for this molecule:
 - [AiZynthFinder] This molecule was solved by AiZynthFinder. See predicted routes: Cached predicted routes are not available yet.
 - [SynPlanner] This molecule was solved by SynPlanner. See predicted routes: Cached predicted routes are not available yet.


## 3. Predict

Predict a route computationally. Results are cached in memory for this session (`cache=True` by default) so a later Search this session surfaces them too — but the cache disappears when the notebook restarts; it is never written to the CSV. Uses a stub `CASPEngine` here — swap in a real open-source CASP tool (e.g. AiZynthFinder) via the same `predict_route` interface.

In [5]:
result = app.predict_casp_solvability("C(O)(C(O)=O)C(C1C=CC=CC=1)NC(C1C=CC=CC=1)=O")
print(result.to_dataframe().to_string())

            tool probability                                                             url
0  aizynthfinder         84%                    https://github.com/MolecularAI/aizynthfinder
1     synplanner         82%  https://github.com/Laboratoire-de-Chemoinformatique/SynPlanner


### 4. Monitor

Fetch recent papers, score with a classifier, display ranked. This is **display-only** - nothing here is written into the dataset. If a candidate turns out to be a real route, the way to record it is to add a row to the CSV and reload.

In [6]:
result  = app.monitor(year_min=1990, year_max=2025)
print(result.message)

16878 paper(s) for 1990-2025, sorted by predicted route probability.


In [7]:
result.to_dataframe()

,route_prob,journal,title,publication_date,doi
0,83%,Organic Process Research & Development,Practical Synthesis of a HIV Integrase Inhibitor,29/10/2008,10.1021/op800153y
1,82%,Tetrahedron,An expeditious route to the synthesis of adeno...,01/05/1996,10.1016/0040-4039(96)00632-6
2,81%,Organic Process Research & Development,Development of a Scalable Route to the SMO Rec...,31/10/2012,10.1021/op300170q
3,81%,Tetrahedron,An efficient route for synthesis of spirocycli...,14/08/2024,10.1016/j.tetlet.2024.155250
4,81%,Tetrahedron,Synthesis of combretastatin D-2. An efficient ...,01/06/1994,10.1016/s0040-4039(00)73369-7
...,...,...,...,...,...
16873,58%,Synlett,Extending the Utility of the Bartoli Indolizat...,23/01/2013,10.1055/s-0032-1318137
16874,58%,Angewandte Chemie International Edition,Catalytic Asymmetric Total Synthesis of <i>ent...,08/01/2010,10.1002/anie.200906678
16875,58%,Journal of Organic Chemistry,Modular and Stereodivergent Approach to Unbran...,26/08/2016,10.1021/acs.joc.6b01051
16876,58%,Organic Letters,Asymmetric Total Synthesis of (−)-Spirofungin ...,09/11/2005,10.1021/ol052039k


### 5. Download

Export the dataset (or a filtered slice) as a flat table for ML training. Every row comes from the CSV — Hunter output never appears here since it's never written into the dataset.

In [8]:
app.download()

,inchikey,canonical_smiles,doi,title,abstract,journal,year,source
0,KJHKTHWMRKYKJE-SUGCFTRWSA-N,Cc1cccc(C)c1OCC(=O)N[C@@H](Cc1ccccc1)[C@@H](O)...,10.1021/op990202j,Synthesis of HIV Protease Inhibitor ABT-378 (L...,A large scale process for the synthesis of HIV...,Organic Process Research & Development,2000,seed
1,XOEMATDHVZOBSG-UHFFFAOYSA-N,C#CCOc1cc(-n2nc3n(c2=O)CCCC3)c(Cl)cc1Cl,10.1021/op9901994,Discovery and Development of a Commercial Synt...,A commercial synthesis of the DuPont herbicide...,Organic Process Research & Development,2001,seed
2,ILMMRHUILQOQGP-UHFFFAOYSA-N,CC(C)(C)c1cc(CC2SCNC2=O)cc(C(C)(C)C)c1O,10.1021/op990197j,The Development of a Manufacturable Synthesis ...,The development of a manufacturable synthesis ...,Organic Process Research & Development,2000,seed
3,MMNZCESXFOCVMF-UHFFFAOYSA-N,O=C1C(C(O)COc2ccc(F)cc2)C(c2ccc(O)cc2)N1c1ccc(...,10.1021/op990196r,A Concise Asymmetric Synthesis of A β-Lactam-B...,"A concise, four-step, asymmetric synthesis of ...",Organic Process Research & Development,2000,seed
4,MRLGCTNJRREZHZ-UHFFFAOYSA-N,O=Cc1cccc(Oc2ccccc2)c1,10.1021/op9901947,Use of Sodium Bromate for Aromatic Bromination...,Sodium bromate is a powerful brominating agent...,Organic Process Research & Development,1999,seed
...,...,...,...,...,...,...,...,...
1475,CGWKMDYVWRDDRF-RBUKOAKNSA-N,CC[C@]12CCCN[C@H]1n1c(c(CCO)c3ccccc31)CC2,10.1002/anie.201812822,Scalable Enantioselective Total Synthesis of (...,A scalable enantioselective total synthesis of...,Angewandte Chemie International Edition,2018,seed
1476,LQBVNQSMGBZMKD-UHFFFAOYSA-N,CC1(C)CCC(CN2CCN(c3ccc(C(=O)NS(=O)(=O)c4ccc(NC...,10.1021/acs.joc.8b02750,Development of a Convergent Large-Scale Synthe...,The process development of a new synthetic rou...,Journal of Organic Chemistry,2019,seed
1477,MAOIDRRXRLYJNV-NRFANRHFSA-N,CC[C@](O)(c1nnc(NCc2ccc3c(-c4ccc(F)cc4)cc(=O)o...,10.1021/jo100561u,A Practical Synthesis of 5-Lipoxygenase Inhibi...,"Practical, chromatography-free syntheses of 5-...",Journal of Organic Chemistry,2010,seed
1478,UVBUBMSSQKOIBE-DSLOAKGESA-N,CCCC[C@@H](C)[C@@H](OC(=O)C[C@@H](CC(=O)O)C(=O...,10.1021/ja9009265,Total Synthesis of the Sphingolipid Biosynthes...,The first total synthesis of the sphingolipid ...,Journal of the American Chemical Society,2009,seed
